# 🛑 Cease & Desist Document Processing System

**Multi-Agent AI Pipeline** using LangChain + LangGraph + Groq LLM

---

### 🔄 Flow
```
PDF Upload → Text Extraction → Classification Agent
                                      ↓
              ┌───────────────────────┼───────────────────────┐
           Cease                  Uncertain               Irrelevant
              ↓                      ↓                        ↓
        Database Agent          HITL Review             Archive Agent
              └──────────────────────┴────────────────────────┘
                                      ↓
                                 Audit Agent
```

### 📋 Classifications
| Label | Meaning | Action |
|-------|---------|--------|
| ✅ **Cease** | Valid cease & desist request | Save to SQLite DB |
| ⚠️ **Uncertain** | Needs human review | HITL widget |
| ❌ **Irrelevant** | Not a cease request | Archive to CSV |

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q langchain langchain-groq langchain-community langgraph langsmith pypdf python-dotenv ipywidgets

## 🔑 Step 2 — API Keys

> **Recommended:** Use Colab Secrets (🔑 icon in left sidebar) and add `GROQ_API_KEY` and `LANGCHAIN_API_KEY`.
>
> Or paste them directly below.

In [ ]:
import os

# --- Option A: Colab Secrets (recommended) ---
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"]        = userdata.get("GROQ_API_KEY")
    os.environ["LANGCHAIN_API_KEY"]   = userdata.get("LANGCHAIN_API_KEY")
except Exception:
    pass

# --- Option B: Paste directly (fallback) ---
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"]      = "YOUR_GROQ_API_KEY_HERE"
    os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGCHAIN_API_KEY_HERE"

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "cease-desist-processor"

print("✅ API keys configured.")

✅ API keys configured.


## 📄 Step 3 — Upload PDF Documents

Upload one or more PDF files. They will be saved to `/content/pdfs/`.

In [ ]:
import os
from pathlib import Path

PDF_DIR = Path("/content/pdfs")
PDF_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files
    print("📂 Select your PDF files to upload:")
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = PDF_DIR / name
        dest.write_bytes(data)
        print(f"  ✅ Saved: {name}")
except Exception:
    # Running locally — place PDFs in /content/pdfs manually
    print("ℹ️  Not in Colab. Place PDFs in:", PDF_DIR)

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"\n📋 {len(pdfs)} PDF(s) ready: {[p.name for p in pdfs]}")

📂 Select your PDF files to upload:


Saving bw_doc_1.pdf to bw_doc_1.pdf
Saving bw_doc_2.pdf to bw_doc_2.pdf
Saving bw_doc_3.pdf to bw_doc_3.pdf
Saving bw_doc_4.pdf to bw_doc_4.pdf
Saving bw_doc_5.pdf to bw_doc_5.pdf
Saving LoA1.pdf to LoA1.pdf
Saving LOA2.pdf to LOA2.pdf
Saving LOA3.pdf to LOA3.pdf
Saving LOA4.pdf to LOA4.pdf
Saving LOA5.pdf to LOA5.pdf
Saving LOA6.pdf to LOA6.pdf
Saving LOA7.pdf to LOA7.pdf
Saving LOA8.pdf to LOA8.pdf
Saving LOA9.pdf to LOA9.pdf
Saving notice_1.pdf to notice_1.pdf
Saving notice_2.pdf to notice_2.pdf
Saving notice_3.pdf to notice_3.pdf
Saving notice_4.pdf to notice_4.pdf
Saving notice_5.pdf to notice_5.pdf
  ✅ Saved: bw_doc_1.pdf
  ✅ Saved: bw_doc_2.pdf
  ✅ Saved: bw_doc_3.pdf
  ✅ Saved: bw_doc_4.pdf
  ✅ Saved: bw_doc_5.pdf
  ✅ Saved: LoA1.pdf
  ✅ Saved: LOA2.pdf
  ✅ Saved: LOA3.pdf
  ✅ Saved: LOA4.pdf
  ✅ Saved: LOA5.pdf
  ✅ Saved: LOA6.pdf
  ✅ Saved: LOA7.pdf
  ✅ Saved: LOA8.pdf
  ✅ Saved: LOA9.pdf
  ✅ Saved: notice_1.pdf
  ✅ Saved: notice_2.pdf
  ✅ Saved: notice_3.pdf
  ✅ Saved: notic

## ⚙️ Step 4 — System Setup (DB, Paths, LLM)

In [ ]:
import sqlite3
import csv
import json
from datetime import datetime
from typing import TypedDict, Literal, Optional

from pypdf import PdfReader
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Paths
BASE_DIR  = Path("/content")
DB_PATH   = BASE_DIR / "cease_desist.db"
ARCHIVE   = BASE_DIR / "archive.csv"
AUDIT_LOG = BASE_DIR / "audit.log"

# LLM
LLM = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

# Database
def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS cease_requests (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            received_at TEXT,
            doc_name    TEXT,
            details     TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()
print("✅ LLM, DB, and paths initialized.")

✅ LLM, DB, and paths initialized.


## 🧩 Step 5 — Graph State & Tools

In [ ]:
# ── State ──────────────────────────────────────────────────────────────────
class DocState(TypedDict):
    doc_name:       str
    doc_text:       str
    classification: Optional[Literal["Cease", "Uncertain", "Irrelevant"]]
    explanation:    Optional[str]
    hitl_decision:  Optional[Literal["Cease", "Irrelevant"]]
    details:        Optional[str]
    audit_entries:  list

# ── Tools ──────────────────────────────────────────────────────────────────
@tool
def load_pdf(path: str) -> str:
    """Extract text from a PDF file."""
    reader = PdfReader(path)
    text = "\n".join(p.extract_text() or "" for p in reader.pages).strip()
    return text or f"[SCANNED IMAGE PDF - no text layer. File: {Path(path).name}]"

@tool
def save_to_database(doc_name: str, details: str) -> str:
    """Save a Cease request to SQLite."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT INTO cease_requests (received_at, doc_name, details) VALUES (?,?,?)",
        (datetime.now().isoformat(), doc_name, details),
    )
    conn.commit()
    conn.close()
    return f"Saved '{doc_name}' to database."

@tool
def archive_document(doc_name: str) -> str:
    """Archive an irrelevant document to CSV."""
    write_header = not ARCHIVE.exists()
    with open(ARCHIVE, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["received_at", "doc_name"])
        w.writerow([datetime.now().isoformat(), doc_name])
    return f"Archived '{doc_name}'."

@tool
def write_audit(doc_name: str, classification: str, explanation: str, action: str) -> str:
    """Append an entry to the audit log."""
    entry = (
        f"[{datetime.now().isoformat()}] "
        f"DOC={doc_name} | CLASS={classification} | ACTION={action} | {explanation}"
    )
    with open(AUDIT_LOG, "a", encoding="utf-8") as f:
        f.write(entry + "\n")
    return entry

print("✅ State and tools defined.")

✅ State and tools defined.


## 🤖 Step 6 — Agent Nodes

In [ ]:
# ── Classification Agent ───────────────────────────────────────────────────
def classification_agent(state: DocState) -> DocState:
    if state.get("classification"):
        print(f"  [Classifier] {state['doc_name']} → {state['classification']} (pre-set)")
        return state

    system = SystemMessage(content="""You are a multilingual legal document classifier.
Classify the document into exactly one of:
- "Cease"      : A valid Cease & Desist request to stop communication
- "Uncertain"  : Possibly a cease request but unclear; needs human review
- "Irrelevant" : Not a cease & desist request

Respond ONLY with valid JSON:
{"classification": "<Cease|Uncertain|Irrelevant>", "explanation": "<brief reason>", "details": "<key extracted info>"}
""")
    human = HumanMessage(content=f"Document: {state['doc_name']}\n\n{state['doc_text'][:4000]}")
    response = LLM.invoke([system, human])

    try:
        raw = response.content.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        data = json.loads(raw)
    except Exception:
        c = response.content
        label = "Cease" if ("Cease" in c and "Uncertain" not in c) else ("Uncertain" if "Uncertain" in c else "Irrelevant")
        data = {"classification": label, "explanation": c[:200], "details": ""}

    details = data.get("details", "")
    if not isinstance(details, str):
        details = json.dumps(details)

    print(f"  [Classifier] {state['doc_name']} → {data['classification']}")
    return {**state, "classification": data["classification"], "explanation": data["explanation"], "details": details}


# ── Database Agent ─────────────────────────────────────────────────────────
def database_agent(state: DocState) -> DocState:
    details = state["details"] or ""
    result = save_to_database.invoke({"doc_name": state["doc_name"], "details": details})
    print(f"  [DB Agent] {result}")
    return {**state, "audit_entries": state["audit_entries"] + [result]}


# ── Archiving Agent ────────────────────────────────────────────────────────
def archiving_agent(state: DocState) -> DocState:
    result = archive_document.invoke({"doc_name": state["doc_name"]})
    print(f"  [Archive Agent] {result}")
    return {**state, "audit_entries": state["audit_entries"] + [result]}


# ── Audit Agent ────────────────────────────────────────────────────────────
def audit_agent(state: DocState) -> DocState:
    hitl   = state.get("hitl_decision")
    action = ("HITL-" + hitl) if hitl else state["classification"]
    entry  = write_audit.invoke({
        "doc_name":       state["doc_name"],
        "classification": state["classification"],
        "explanation":    state["explanation"] or "",
        "action":         action,
    })
    print(f"  [Audit] Logged.")
    return {**state, "audit_entries": state["audit_entries"] + [entry]}


print("✅ Agent nodes defined.")

✅ Agent nodes defined.


## 🙋 Step 7 — HITL Agent (Interactive Widget)

For **Uncertain** documents, a review widget will appear asking for your decision.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def hitl_agent(state: DocState) -> DocState:
    """Human-in-the-Loop review using interactive widgets."""
    print(f"\n{'─'*60}")
    print(f"⚠️  HUMAN REVIEW REQUIRED")
    print(f"   Document : {state['doc_name']}")
    print(f"   Reason   : {state['explanation']}")
    print(f"   Preview  : {state['doc_text'][:300]}")
    print(f"{'─'*60}")

    decision_holder = {"value": None}

    btn_cease      = widgets.Button(description="✅ Cease",      button_style="success", layout=widgets.Layout(width="140px"))
    btn_irrelevant = widgets.Button(description="❌ Irrelevant", button_style="danger",  layout=widgets.Layout(width="140px"))
    btn_skip       = widgets.Button(description="⏭ Skip",       button_style="warning", layout=widgets.Layout(width="140px"))
    output         = widgets.Output()

    def on_click(btn):
        label_map = {btn_cease: "Cease", btn_irrelevant: "Irrelevant", btn_skip: None}
        decision_holder["value"] = label_map[btn]
        with output:
            print(f"  → Decision recorded: {decision_holder['value'] or 'Skip'}")
        for b in [btn_cease, btn_irrelevant, btn_skip]:
            b.disabled = True

    btn_cease.on_click(on_click)
    btn_irrelevant.on_click(on_click)
    btn_skip.on_click(on_click)

    display(widgets.HBox([btn_cease, btn_irrelevant, btn_skip]), output)

    # Wait for decision
    import time
    while decision_holder["value"] is None and not btn_cease.disabled:
        time.sleep(0.5)

    chosen = decision_holder["value"]
    if chosen is None:
        return {**state, "hitl_decision": None}

    return {**state, "hitl_decision": chosen, "classification": chosen}

print("✅ HITL widget agent defined.")

✅ HITL widget agent defined.


## 🔀 Step 8 — Build LangGraph Pipeline

In [ ]:
def route_after_classification(state: DocState) -> str:
    return {"Cease": "database", "Irrelevant": "archive"}.get(state["classification"], "hitl")

def route_after_hitl(state: DocState) -> str:
    return {"Cease": "database", "Irrelevant": "archive"}.get(state.get("hitl_decision"), "audit")


def build_graph():
    g = StateGraph(DocState)

    g.add_node("classify", classification_agent)
    g.add_node("database", database_agent)
    g.add_node("archive",  archiving_agent)
    g.add_node("hitl",     hitl_agent)
    g.add_node("audit",    audit_agent)

    g.set_entry_point("classify")

    g.add_conditional_edges("classify", route_after_classification,
                            {"database": "database", "archive": "archive", "hitl": "hitl"})
    g.add_conditional_edges("hitl", route_after_hitl,
                            {"database": "database", "archive": "archive", "audit": "audit"})
    g.add_edge("database", "audit")
    g.add_edge("archive",  "audit")
    g.add_edge("audit",    END)

    return g.compile(checkpointer=MemorySaver())


graph = build_graph()
print("✅ LangGraph pipeline compiled.")

✅ LangGraph pipeline compiled.


## ▶️ Step 9 — Run the Pipeline

Processes all uploaded PDFs one by one. Uncertain documents will show a review widget.

In [ ]:
def process_document(pdf_path: Path, thread_id: str):
    print(f"\n{'═'*60}")
    print(f"📄 Processing: {pdf_path.name}")
    print(f"{'═'*60}")

    text       = load_pdf.invoke({"path": str(pdf_path)})
    is_scanned = text.startswith("[SCANNED IMAGE PDF")

    if is_scanned:
        print("  [Loader] Scanned image PDF — auto-classifying as Irrelevant.")

    initial_state: DocState = {
        "doc_name":       pdf_path.name,
        "doc_text":       text,
        "classification": "Irrelevant" if is_scanned else None,
        "explanation":    "Scanned image PDF with no extractable text" if is_scanned else None,
        "hitl_decision":  None,
        "details":        None,
        "audit_entries":  [],
    }

    final = graph.invoke(initial_state, config={"configurable": {"thread_id": thread_id}})
    print(f"  ✅ Result → {final['classification']}")
    return final


# ── Run all PDFs ────────────────────────────────────────────────────────────
pdfs = sorted(PDF_DIR.glob("*.pdf"))

if not pdfs:
    print("⚠️  No PDFs found. Please run Step 3 to upload files.")
else:
    print(f"🚀 Starting pipeline — {len(pdfs)} document(s) to process.\n")
    results = []
    for i, pdf in enumerate(pdfs):
        results.append(process_document(pdf, thread_id=f"doc-{i}"))

    print(f"\n{'═'*60}")
    print("🏁 All documents processed!")

🚀 Starting pipeline — 19 document(s) to process.


════════════════════════════════════════════════════════════
📄 Processing: LOA2.pdf
════════════════════════════════════════════════════════════


/usr/local/lib/python3.12/dist-packages/langsmith/client.py:538: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


  [Classifier] LOA2.pdf → Cease
  [DB Agent] Saved 'LOA2.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA3.pdf
════════════════════════════════════════════════════════════


  [Classifier] LOA3.pdf → Cease
  [DB Agent] Saved 'LOA3.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA4.pdf
════════════════════════════════════════════════════════════
  [Classifier] LOA4.pdf → Cease
  [DB Agent] Saved 'LOA4.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA5.pdf
════════════════════════════════════════════════════════════


  [Classifier] LOA5.pdf → Cease
  [DB Agent] Saved 'LOA5.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA6.pdf
════════════════════════════════════════════════════════════


  [Classifier] LOA6.pdf → Cease
  [DB Agent] Saved 'LOA6.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA7.pdf
════════════════════════════════════════════════════════════


  [Classifier] LOA7.pdf → Cease
  [DB Agent] Saved 'LOA7.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA8.pdf
════════════════════════════════════════════════════════════
  [Classifier] LOA8.pdf → Cease
  [DB Agent] Saved 'LOA8.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LOA9.pdf
════════════════════════════════════════════════════════════
  [Classifier] LOA9.pdf → Cease
  [DB Agent] Saved 'LOA9.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: LoA1.pdf
════════════════════════════════════════════════════════════


  [Classifier] LoA1.pdf → Cease
  [DB Agent] Saved 'LoA1.pdf' to database.
  [Audit] Logged.
  ✅ Result → Cease

════════════════════════════════════════════════════════════
📄 Processing: bw_doc_1.pdf
════════════════════════════════════════════════════════════
  [Loader] Scanned image PDF — auto-classifying as Irrelevant.
  [Classifier] bw_doc_1.pdf → Irrelevant (pre-set)
  [Archive Agent] Archived 'bw_doc_1.pdf'.
  [Audit] Logged.
  ✅ Result → Irrelevant

════════════════════════════════════════════════════════════
📄 Processing: bw_doc_2.pdf
════════════════════════════════════════════════════════════
  [Loader] Scanned image PDF — auto-classifying as Irrelevant.
  [Classifier] bw_doc_2.pdf → Irrelevant (pre-set)
  [Archive Agent] Archived 'bw_doc_2.pdf'.
  [Audit] Logged.
  ✅ Result → Irrelevant

════════════════════════════════════════════════════════════
📄 Processing: bw_doc_3.pdf
════════════════════════════════════════════════════════════
  [Loader] Scanned image PDF — auto-clas

## 📊 Step 10 — View Results

In [ ]:
import pandas as pd

# ── Summary Table ───────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {"Document": r["doc_name"], "Classification": r["classification"], "Explanation": r["explanation"]}
    for r in results
])
print("📋 Processing Summary")
display(summary)

# ── Cease Requests from DB ──────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
db_df = pd.read_sql("SELECT * FROM cease_requests", conn)
conn.close()
print(f"\n🗄️  Cease Requests in Database ({len(db_df)} records)")
display(db_df)

# ── Archived Documents ──────────────────────────────────────────────────────
if ARCHIVE.exists():
    archive_df = pd.read_csv(ARCHIVE)
    print(f"\n📁 Archived Documents ({len(archive_df)} records)")
    display(archive_df)

# ── Audit Log ───────────────────────────────────────────────────────────────
if AUDIT_LOG.exists():
    print("\n📝 Audit Log")
    print(AUDIT_LOG.read_text(encoding="utf-8"))

📋 Processing Summary


,Document,Classification,Explanation
0,LOA2.pdf,Cease,The document explicitly states 'Cease and desi...
1,LOA3.pdf,Cease,The document explicitly states 'Cease and desi...
2,LOA4.pdf,Cease,The document explicitly states 'Cease and desi...
3,LOA5.pdf,Cease,The document explicitly states 'Cease and desi...
4,LOA6.pdf,Cease,The document explicitly states 'Cese y desista...
5,LOA7.pdf,Cease,The document contains a clear instruction to c...
6,LOA8.pdf,Cease,The document contains a clear instruction to c...
7,LOA9.pdf,Cease,The document contains a clear instruction to c...
8,LoA1.pdf,Cease,The document explicitly states 'Cease and desi...
9,bw_doc_1.pdf,Irrelevant,Scanned image PDF with no extractable text



🗄️  Cease Requests in Database (9 records)


,id,received_at,doc_name,details
0,1,2026-03-27T02:17:26.280800,LOA2.pdf,"{""account_holder"": ""LOVETTA CANNUNZIATA"", ""age..."
1,2,2026-03-27T02:17:26.914701,LOA3.pdf,"{""account_holder"": ""BELLINA DAVANZO AMOEDO, RA..."
2,3,2026-03-27T02:17:27.399468,LOA4.pdf,"{""law_firm"": ""Five Lakes Law Group PLLC"", ""acc..."
3,4,2026-03-27T02:17:27.966138,LOA5.pdf,"{""account_holder"": ""RANDLE L WIDHALM"", ""agent""..."
4,5,2026-03-27T02:17:28.873845,LOA6.pdf,"{""law_firm"": ""Five Lakes Law Group PLLC"", ""cli..."
5,6,2026-03-27T02:17:30.283725,LOA7.pdf,"{""name"": ""LAW OFFICES OF DONALD A GREEN, APLC""..."
6,7,2026-03-27T02:17:30.929377,LOA8.pdf,"{""cease instruction"": ""F ennore, I (We) formal..."
7,8,2026-03-27T02:17:31.690156,LOA9.pdf,"{""cease instruction"": ""Furthermore; I (We) for..."
8,9,2026-03-27T02:17:32.143725,LoA1.pdf,"{""account_holder"": ""DAVANZO.BELLINA AMOEDO,RAD..."



📁 Archived Documents (10 records)


,received_at,doc_name
0,2026-03-27T02:17:32.168719,bw_doc_1.pdf
1,2026-03-27T02:17:32.182509,bw_doc_2.pdf
2,2026-03-27T02:17:32.200307,bw_doc_3.pdf
3,2026-03-27T02:17:32.213817,bw_doc_4.pdf
4,2026-03-27T02:17:32.229300,bw_doc_5.pdf
5,2026-03-27T02:17:32.250177,notice_1.pdf
6,2026-03-27T02:17:32.268663,notice_2.pdf
7,2026-03-27T02:17:32.284846,notice_3.pdf
8,2026-03-27T02:17:32.302495,notice_4.pdf
9,2026-03-27T02:17:32.316626,notice_5.pdf



📝 Audit Log
[2026-03-27T02:17:26.293990] DOC=LOA2.pdf | CLASS=Cease | ACTION=Cease | The document explicitly states 'Cease and desist all communications regarding this account' and instructs to direct all communications to the appointed agent.
[2026-03-27T02:17:26.927910] DOC=LOA3.pdf | CLASS=Cease | ACTION=Cease | The document explicitly states 'Cease and desist all communications regarding this account'
[2026-03-27T02:17:27.411789] DOC=LOA4.pdf | CLASS=Cease | ACTION=Cease | The document explicitly states 'Cease and desist all communications regarding this account'
[2026-03-27T02:17:27.976979] DOC=LOA5.pdf | CLASS=Cease | ACTION=Cease | The document explicitly states 'Cease and desist all communications regarding this account' and instructs to direct all communications to the appointed agent.
[2026-03-27T02:17:28.887347] DOC=LOA6.pdf | CLASS=Cease | ACTION=Cease | The document explicitly states 'Cese y desista de todas las comunicaciones relacionadas con esta cuenta' which translate

## 💾 Step 11 — Download Output Files

In [ ]:
try:
    from google.colab import files
    for path in [DB_PATH, ARCHIVE, AUDIT_LOG]:
        if path.exists():
            files.download(str(path))
            print(f"⬇️  Downloading: {path.name}")
except Exception:
    print("ℹ️  Not in Colab. Files are at:")
    for path in [DB_PATH, ARCHIVE, AUDIT_LOG]:
        print(f"   {path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: cease_desist.db


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: archive.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: audit.log
